# TP 2 — Volatilité, VaR et backtesting

**Séance 3 · 50 minutes · en binômes**


### Objectifs

Construire une chaîne complète de gestion du risque, **et la valider** :

1. Trois estimateurs de volatilité autour d'un choc
2. Estimation du facteur de décroissance $\lambda$ — comparaison à 0,94
3. Quatre VaR en fenêtre glissante
4. Expected Shortfall
5. Graphique des dépassements
6. **Kupiec + Christoffersen → tableau de verdict**
7. Corrélations glissantes
8. **Rapport de risque — 2 pages**

### Convention du cours

La VaR est un **nombre positif** représentant une perte.
`VaR = -np.quantile(r, alpha)`


---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# retrouve outils_cours.py, que le notebook soit ouvert depuis 02-TP/ ou 02-TP/corriges/
for _cand in (Path.cwd(), *list(Path.cwd().parents)[:3]):
    if (_cand / "outils_cours.py").exists():
        sys.path.insert(0, str(_cand))
        break
import outils_cours as oc

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": .3})
np.random.seed(2026)
print("Environnement prêt.")

prix = oc.charger_panier()
r = oc.rendements_log(prix)

ACTIF = "Bitcoin"       # <- changez-le pour comparer
ALPHA = 0.01            # seuil de VaR
x = r[ACTIF].dropna()
print(f"{ACTIF} : {len(x)} rendements, de {x.index.min():%Y-%m-%d} à {x.index.max():%Y-%m-%d}")

## 1. Trois estimateurs de volatilité

Volatilité réalisée sur échantillon complet, glissante (21 et 252 jours), EWMA.

In [ ]:
vol_pleine = x.std(ddof=1)
vol_21 = x.rolling(21).std(ddof=1)
vol_252 = x.rolling(252).std(ddof=1)
vol_ewma = np.sqrt(oc.ewma_variance(x, lam=0.94))

fig, ax = plt.subplots(figsize=(12, 5))
ann = np.sqrt(365)
(vol_21 * ann).plot(ax=ax, lw=.8, alpha=.65, label="glissante 21 j")
(vol_252 * ann).plot(ax=ax, lw=1.6, label="glissante 252 j")
(vol_ewma * ann).plot(ax=ax, lw=1.3, label="EWMA λ=0,94")
ax.axhline(vol_pleine * ann, color="k", ls="--", lw=1,
           label=f"réalisée pleine fenêtre ({vol_pleine*ann:.0%})")
ax.set_title(f"{ACTIF} — volatilité annualisée (√365) selon l'estimateur")
ax.set_ylabel("volatilité annualisée")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

**Question 1.1** — Repérez sur le graphique un décrochage brutal de la
volatilité à 252 jours **sans événement de marché ce jour-là**. Combien de temps
après le choc initial survient-il ? Comment s'appelle cet artefact ?

**Question 1.2** — La volatilité réalisée sur échantillon complet (ligne noire)
décrit-elle une période réellement vécue ?

*Vos réponses :*

>

## 2. Estimer λ plutôt que de le supposer

$\lambda = 0{,}94$ vient de RiskMetrics (1996), calibré sur des actions, taux et
changes — sans aucune crypto. Estimons-le sur nos données, en minimisant
l'erreur quadratique de prévision de $r_t^2$.

In [ ]:
# À COMPLÉTER
# 1) écrire une fonction erreur_prevision(lam, serie) qui renvoie l'erreur
#    quadratique moyenne entre la variance EWMA et le carré du rendement,
#    en ignorant les 60 premières observations (rodage) ;
# 2) la minimiser sur une grille de lambda entre 0,80 et 0,99 ;
# 3) tracer la courbe et comparer le lambda optimal à 0,94 ;
# 4) convertir les deux lambdas en DEMI-VIE (ln(0,5)/ln(lambda)).


**Question 2.1** — Votre $\lambda$ optimal est-il supérieur ou inférieur à 0,94 ?
Traduisez cela en termes de demi-vie, puis en une phrase de gestionnaire de risque.

*Votre réponse :*

>

## 3. Quatre VaR en fenêtre glissante

Toutes les VaR sont calculées **avec la seule information disponible en $t-1$**.
Sans cela, le backtesting n'a aucun sens.

In [ ]:
FENETRE = 500

var_hist, var_gauss, var_stud, var_mc = {}, {}, {}, {}
vol_e = np.sqrt(oc.ewma_variance(x, lam=lam_opt))

for i in range(FENETRE, len(x)):
    date = x.index[i]
    passe = x.iloc[i - FENETRE:i]                 # strictement antérieur à t
    mu, sd = passe.mean(), passe.std(ddof=1)
    sd_ewma = float(vol_e.iloc[i])                # déjà décalé d'un cran

    var_hist[date] = oc.var_historique(passe, ALPHA)
    var_gauss[date] = oc.var_gaussienne(mu, sd_ewma, ALPHA)
    nu_i = max(stats.t.fit((passe - mu) / sd, floc=0, fscale=1)[0], 2.5)
    var_stud[date] = oc.var_student(mu, sd_ewma, nu_i, ALPHA)
    var_mc[date] = oc.var_monte_carlo(mu, sd_ewma, ALPHA, n_sim=20_000,
                                      loi="student", nu=nu_i, graine=i)

VAR = pd.DataFrame({
    "historique": pd.Series(var_hist),
    "gaussienne (EWMA)": pd.Series(var_gauss),
    "Student (EWMA)": pd.Series(var_stud),
    "Monte Carlo (t)": pd.Series(var_mc),
})
print(f"{len(VAR)} jours backtestés, de {VAR.index.min():%Y-%m-%d} à {VAR.index.max():%Y-%m-%d}")
(VAR.describe().T[["mean", "min", "max"]] * 100).round(2)

**Question 3.1** — Classez les quatre méthodes de la plus conservatrice à la
plus permissive **en moyenne**. Le classement est-il stable dans le temps ?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
(VAR * 100).plot(ax=ax, lw=1.1)
ax.set_title(f"{ACTIF} — VaR à {ALPHA:.0%}, quatre méthodes, fenêtre glissante {FENETRE} j")
ax.set_ylabel("perte quotidienne, en %")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print("VaR moyenne sur la période :")
print((VAR.mean() * 100).round(2).sort_values(ascending=False).to_string())

## 4. Expected Shortfall

La perte moyenne **au-delà** de la VaR — l'information que la VaR ne donne pas.

In [ ]:
# À COMPLÉTER
# Sur les FENETRE derniers jours : calculer la VaR historique et l'ES historique
# (oc.var_historique, oc.expected_shortfall_historique), puis le rapport ES/VaR.
# Comparer à la valeur attendue sous normalité (≈ 1,15 à alpha = 1 %).


## 5. Dépassements et backtesting

Le cœur du TP. Un modèle de risque qui n'est pas backtesté n'est pas un modèle.

In [ ]:
r_bt = x.loc[VAR.index]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(r_bt.index, r_bt * 100, lw=.6, color="0.55", label="rendement quotidien")
ax.plot(VAR.index, -VAR["gaussienne (EWMA)"] * 100, lw=1.2,
        color="tab:orange", label="−VaR gaussienne")
ax.plot(VAR.index, -VAR["Student (EWMA)"] * 100, lw=1.2,
        color="tab:blue", label="−VaR Student")

dep = r_bt[(-r_bt) > VAR["gaussienne (EWMA)"]]
ax.scatter(dep.index, dep * 100, color="crimson", s=16, zorder=5,
           label=f"dépassements gaussienne (n={len(dep)})")
ax.set_title(f"{ACTIF} — dépassements de VaR à {ALPHA:.0%}")
ax.set_ylabel("rendement, en %")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

**Question 5.1** — Les dépassements de la VaR gaussienne sont-ils dispersés
dans le temps, ou concentrés sur quelques périodes ? Quel test formalise cette
question ?

**Question 5.2 — LA question du TP.** Backtestez les quatre méthodes avec
Kupiec et Christoffersen, et produisez un tableau de verdict.

In [ ]:
# À COMPLÉTER
# Pour chacune des quatre colonnes de VAR, lancer oc.backtest_complet
# et assembler les résultats dans un DataFrame indexé par le nom du modèle.
# (penser à retirer la colonne technique "_depassements")


**Question 5.3** — Lisez le tableau :

- Quelle(s) méthode(s) passe(nt) le test de Kupiec ?
- Quelle(s) méthode(s) passe(nt) le test d'indépendance ?
- Une méthode peut-elle passer Kupiec et échouer à Christoffersen ? Qu'est-ce que
  cela signifie **concrètement** pour un gérant de risque ?

*Vos réponses :*

>

**Question 5.4** — Refaites le backtesting à $\alpha = 5\%$.
Les conclusions changent-elles ? Pourquoi la puissance des tests est-elle
différente ?

In [ ]:
# À COMPLÉTER
# Refaire le calcul des VaR et le backtesting à alpha = 5 %.
# Commenter : à 5 %, on attend ~5 fois plus de dépassements qu'à 1 %,
# donc le test est PLUS puissant. Vos conclusions changent-elles ?


## 6. Corrélations glissantes : la diversification tient-elle ?

Retour sur le débat de la séance 1.

In [ ]:
FEN_CORR = 90
corr_btc_sp = r["Bitcoin"].rolling(FEN_CORR).corr(r["SP500"])
corr_or_sp = r["Or"].rolling(FEN_CORR).corr(r["SP500"])

fig, ax = plt.subplots(figsize=(12, 4.8))
corr_btc_sp.plot(ax=ax, lw=1.2, label="BTC / S&P 500")
corr_or_sp.plot(ax=ax, lw=1.2, alpha=.75, label="Or / S&P 500")
ax.axhline(0, color="k", lw=.8)
ax.axvline(pd.Timestamp("2024-01-11"), color="crimson", ls="--", lw=1.2,
           label="lancement des ETF BTC au comptant")
ax.set_title(f"Corrélations glissantes ({FEN_CORR} jours) avec le S&P 500")
ax.set_ylabel("corrélation")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

coupure = "2024-01-11"
print(f"corr. BTC/SP500 avant le {coupure} : {r.loc[:coupure,'Bitcoin'].corr(r.loc[:coupure,'SP500']):+.3f}")
print(f"corr. BTC/SP500 après le {coupure} : {r.loc[coupure:,'Bitcoin'].corr(r.loc[coupure:,'SP500']):+.3f}")
print(f"corr. Or/SP500  avant             : {r.loc[:coupure,'Or'].corr(r.loc[:coupure,'SP500']):+.3f}")
print(f"corr. Or/SP500  après             : {r.loc[coupure:,'Or'].corr(r.loc[coupure:,'SP500']):+.3f}")

**Question 6.1** — La corrélation BTC / S&P 500 a-t-elle augmenté après
janvier 2024 ? De combien ?

**Question 6.2** — Comparez la corrélation en régime « calme » et en régime
« stress » (définissez le stress par les 10 % de jours où la volatilité du S&P 500
est la plus élevée).

In [ ]:
# À COMPLÉTER
# 1) définir le régime de stress : les 10 % de jours où la volatilité glissante
#    (21 j) du S&P 500 est la plus élevée ;
# 2) comparer, pour BTC / ETH / SOL / Or, la corrélation au S&P 500 en régime
#    calme et en régime de stress ;
# 3) MENTIONNER le biais de sélection de Boyer, Gibson & Loretan (1999).


---

## 7. Rapport de risque — à rendre

**2 pages maximum**, en binôme, avec ce notebook exécuté.

Structure attendue :

1. **Cadre** — actif, horizon, seuil, période de backtesting, taille de fenêtre
   et **justification** de chacun de ces choix.
2. **Estimation de la volatilité** — comparaison des trois estimateurs,
   $\lambda$ retenu et pourquoi.
3. **Comparaison des VaR** — le tableau, le graphique des dépassements.
4. **Verdict du backtesting** — le tableau Kupiec / Christoffersen, lu et commenté.
5. **Recommandation** — une méthode, motivée, avec sa principale limite.
6. **Corrélations** — la diversification tient-elle en régime de stress ?

### Barème

| | |
|---|---|
| Exactitude technique | 25 % |
| **Lecture du backtesting** | **30 %** |
| Justification des choix méthodologiques | 20 % |
| Recommandation argumentée + limites | 15 % |
| Reproductibilité | 10 % |

> **Rappel**
>
> Un rapport qui conclut « la VaR Student est la meilleure » sans dire pourquoi ni
> à quelles conditions ne vaut pas mieux qu'un rapport sans conclusion.

---

## 8. Projet final

Sujets, format et grille détaillée : voir `04-Eval/`.
Votre sujet doit être **validé avant la fin de la séance**.
